# Setup

In [1]:
import torch
import torch.nn as nn
import numpy as np
import os
from tqdm import tqdm

WORD_SIZE = 16
MASK_VAL = 2 ** WORD_SIZE - 1
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# Model definition

In [2]:
class ResBlock(nn.Module):
    def __init__(self, channels, kernel_size=3):
        super(ResBlock, self).__init__()
        self.conv1 = nn.Conv1d(channels, 32, kernel_size, padding=1)
        self.bn1 = nn.BatchNorm1d(32)
        self.conv2 = nn.Conv1d(32, channels, kernel_size, padding=1)
        self.bn2 = nn.BatchNorm1d(channels)
        self.relu = nn.ReLU()

    def forward(self, x):
        res = x
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.relu(self.bn2(self.conv2(x)))
        x = x + res
        return x


class ImprovedCNN(nn.Module):
    def __init__(self, depth=10):
        super(ImprovedCNN, self).__init__()
        self.in_channels = 3
        self.res_tower = nn.Sequential(*[ResBlock(self.in_channels) for _ in range(depth)])
        self.fc1 = nn.Linear(3 * 16, 64)
        self.bn1 = nn.BatchNorm1d(64)
        self.fc2 = nn.Linear(64, 64)
        self.bn2 = nn.BatchNorm1d(64)
        self.out = nn.Linear(64, 1)
        self.relu = nn.ReLU()
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.res_tower(x)
        x = x.view(x.size(0), -1)
        x = self.relu(self.bn1(self.fc1(x)))
        x = self.relu(self.bn2(self.fc2(x)))
        x = self.sigmoid(self.out(x))
        return x



# Helpers

In [3]:
def dec_one_round_torch(c0, c1, k):
    """
    Decrypt one round of Speck32/64 using PyTorch tensors.
    """
    if c0.dim() == 1: c0 = c0.unsqueeze(1)
    if c1.dim() == 1: c1 = c1.unsqueeze(1)
    if k.dim() == 1: k = k.unsqueeze(0)


    c1 = c1 ^ c0
    c1 = ((c1 >> 2) | (c1 << 14)) & MASK_VAL


    c0 = c0 ^ k

    if c1.shape != c0.shape:
        c1 = c1.expand_as(c0)

    c0 = (c0 - c1) & MASK_VAL
    c0 = ((c0 << 7) | (c0 >> 9)) & MASK_VAL

    return c0, c1


def extract_features_torch(c0l, c0r, c1l, c1r):
    """
    Extract features for the neural net from ciphertext pairs.
    """
    f1 = c0l ^ c1l
    f2 = c0l ^ c0r
    f3 = c0l ^ c1r

    bits = torch.arange(16, device=c0l.device).reshape(1, 1, 16)
    f_stack = torch.stack([f1, f2, f3], dim=1).unsqueeze(2)
    X = (f_stack >> bits) & 1

    return X.float()


def expand_key(key, rounds):
    """Speck32/64 Key Schedule."""
    k = [key[i] for i in range(4)]
    ks = [0] * rounds
    ks[0] = k[0]
    l = k[1:]

    for i in range(rounds - 1):
        l_curr = l[0]
        l_curr = ((l_curr >> 7) | (l_curr << 9)) & MASK_VAL
        l_curr = (int(l_curr) + int(ks[i])) & MASK_VAL
        l_curr = l_curr ^ i
        l = l[1:] + [l_curr]

        k_curr = ks[i]
        k_curr = ((k_curr << 2) | (k_curr >> 14)) & MASK_VAL
        k_curr = k_curr ^ l_curr
        ks[i + 1] = k_curr

    return ks


def encrypt_vectorized(p_left, p_right, ks):
    """Encrypts a batch of plaintexts using Speck32/64."""
    x = p_left.astype(np.int32)
    y = p_right.astype(np.int32)

    for k in ks:
        x = ((x >> 7) | (x << 9)) & MASK_VAL
        x = (x + y) & MASK_VAL
        x = x ^ k
        y = ((y << 2) | (y >> 14)) & MASK_VAL
        y = y ^ x

    return x.astype(np.uint16), y.astype(np.uint16)


def make_structure(pt0, pt1, diff=(0x0040, 0x0000), neutral_bits=[20, 21, 22, 14, 15]):
    """Gohr's neutral bit structure generation."""
    p0l = np.array([pt0], dtype=np.uint16)
    p0r = np.array([pt1], dtype=np.uint16)

    for i in neutral_bits:
        d = 1 << i
        d_high = (d >> 16) & 0xFFFF
        d_low = d & 0xFFFF
        p0l = np.concatenate([p0l, p0l ^ d_high])
        p0r = np.concatenate([p0r, p0r ^ d_low])

    p1l = p0l ^ diff[0]
    p1r = p0r ^ diff[1]
    return [(p0l, p0r, p1l, p1r)]


def generate_structures(secret_key, rounds, n_structures):
    ks = expand_key(np.frombuffer(secret_key, dtype=np.uint16), rounds)
    structures = []
    for _ in range(n_structures):
        pt0 = np.random.randint(0, 65536, dtype=np.uint16)
        pt1 = np.random.randint(0, 65536, dtype=np.uint16)
        plain_struct = make_structure(pt0, pt1)[0]
        c0l, c0r = encrypt_vectorized(plain_struct[0], plain_struct[1], ks)
        c1l, c1r = encrypt_vectorized(plain_struct[2], plain_struct[3], ks)
        structures.append((c0l, c0r, c1l, c1r))
    return structures



# Attack

In [4]:
def attack_sum_of_logits(ciphertext_structures, model, n_candidates=32):

    model.eval()

    c0l_list = [s[0] for s in ciphertext_structures]
    c0r_list = [s[1] for s in ciphertext_structures]
    c1l_list = [s[2] for s in ciphertext_structures]
    c1r_list = [s[3] for s in ciphertext_structures]

    c0l = torch.tensor(np.concatenate(c0l_list), dtype=torch.int32, device=DEVICE)
    c0r = torch.tensor(np.concatenate(c0r_list), dtype=torch.int32, device=DEVICE)
    c1l = torch.tensor(np.concatenate(c1l_list), dtype=torch.int32, device=DEVICE)
    c1r = torch.tensor(np.concatenate(c1r_list), dtype=torch.int32, device=DEVICE)

    all_keys = torch.arange(0, 65536, dtype=torch.int32, device=DEVICE)
    final_scores = torch.zeros(65536, device=DEVICE)
    key_batch_size = 4096

    with torch.no_grad():
        for i in range(0, 65536, key_batch_size):
            k_batch = all_keys[i: i + key_batch_size]

            d0l, d0r = dec_one_round_torch(c0l, c0r, k_batch)
            d1l, d1r = dec_one_round_torch(c1l, c1r, k_batch)


            X = extract_features_torch(d0l.flatten(), d0r.flatten(), d1l.flatten(), d1r.flatten())
            Z = model(X).squeeze()

            epsilon = 1e-7
            Z = torch.clamp(Z, epsilon, 1 - epsilon)
            logits = torch.log2(Z / (1 - Z))


            logits_matrix = logits.view(c0l.size(0), -1)
            batch_scores = torch.sum(logits_matrix, dim=0)

            final_scores[i: i + key_batch_size] = batch_scores

    top_scores, top_indices = torch.topk(final_scores, n_candidates)
    return list(zip(top_indices.cpu().numpy(), top_scores.cpu().numpy()))



# Evaluation

In [5]:
SCENARIOS = {
5: {"dist_rounds": 5, "attack_rounds": 6, "depth": 10},
6: {"dist_rounds": 6, "attack_rounds": 7, "depth": 10},
7: {"dist_rounds": 7, "attack_rounds": 8, "depth": 1},
8: {"dist_rounds": 8, "attack_rounds": 9, "depth": 1},
}

for r in SCENARIOS:
    cfg = SCENARIOS[r]
    model_path = f"light_models/dnd_speck32_r{r}_d{cfg['depth']}.pt"

    print(f"\nLoading model for {r}-round distinguisher from {model_path}...")

    if not os.path.exists(model_path):
        print(f"Model file not found: {model_path}. Skipping.")
        continue

    try:
        model = ImprovedCNN(depth=cfg['depth']).to(DEVICE)
        checkpoint = torch.load(model_path, map_location=DEVICE, weights_only=False)
        model.load_state_dict(checkpoint['model_state_dict'])
    except Exception as e:
        print(f"Error loading model: {e}")
        continue

    print(f"Starting Attack on {cfg['attack_rounds']} rounds...")

    n_trials = 10
    hits = 0
    ranks = []

    for i in tqdm(range(n_trials), desc="Trials"):
        secret_key = np.frombuffer(os.urandom(8), dtype=np.uint16)
        ks = expand_key(secret_key, cfg['attack_rounds'])
        target_subkey = ks[-1]

        structures = generate_structures(secret_key, cfg['attack_rounds'], n_structures=10)
        candidates = attack_sum_of_logits(structures, model, n_candidates=32)

        predicted_keys = [k for k, score in candidates]
        if target_subkey in predicted_keys:
            hits += 1
            ranks.append(predicted_keys.index(target_subkey) + 1)

    print(f"Results for {cfg['attack_rounds']}-round attack:")
    print(f"  Success Rate: {(hits / n_trials) * 100:.2f}%")
    print(f"  Avg Rank: {np.mean(ranks) if ranks else 0:.2f}")



Loading model for 5-round distinguisher from light_models/dnd_speck32_r5_d10.pt...
Starting Attack on 6 rounds...


Trials: 100%|██████████| 10/10 [42:41<00:00, 256.12s/it]


Results for 6-round attack:
  Success Rate: 100.00%
  Avg Rank: 1.50

Loading model for 6-round distinguisher from light_models/dnd_speck32_r6_d10.pt...
Starting Attack on 7 rounds...


Trials: 100%|██████████| 10/10 [44:01<00:00, 264.18s/it]


Results for 7-round attack:
  Success Rate: 100.00%
  Avg Rank: 1.60

Loading model for 7-round distinguisher from light_models/dnd_speck32_r7_d1.pt...
Starting Attack on 8 rounds...


Trials: 100%|██████████| 10/10 [04:35<00:00, 27.56s/it]


Results for 8-round attack:
  Success Rate: 100.00%
  Avg Rank: 5.10

Loading model for 8-round distinguisher from light_models/dnd_speck32_r8_d1.pt...
Starting Attack on 9 rounds...


Trials: 100%|██████████| 10/10 [04:35<00:00, 27.57s/it]

Results for 9-round attack:
  Success Rate: 10.00%
  Avg Rank: 23.00
